# Imports

In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as rotate

from functions.geon_relations import Object, RectangularPrism
from functions.vectors import cosine_similarity, greatest_landmark_distance, find_center_point_LWLC, find_axis_of_rotation, same_object
from functions.plotting import  add_point, add_landmarks, add_axis, add_object, add_frame, frame_args, \
                                create_data_from_point, create_data_from_landmarks, create_data_from_axis, create_data_from_object

import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as pyo
from plotly.subplots import make_subplots
import math, copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# Model Variables

In [2]:
total_run_time = 0

# timing
production_time = 50                        # 50ms
propositional_difficulty_time = 3           # 1-3 ms
object_encoding_time = 300                      # 150-300ms for each object

# rotation
max_step_size = 30
min_step_size = 1
step_size_decrease = 1/2                    # rate at which step size decreases
decrease_begins = 1/2                       # step size begins to decrease after [decrease_begins] of the total rotation is complete

# decision
difference_confidence = 0
difference_confidence_threshold = 2         # 2-4 repeated steps

# similarity thresholds
landmark_angle_threshold = 0.975
landmark_distance_threshold = 0.3
object_angle_threshold = 0.975
object_distance_threshold = landmark_distance_threshold * 2

# 1. Representation

This is the case I'll model:

![image](test.jpg)

*150 degree diff in pic

In [3]:
# r = rotate.from_euler('z', 60, degrees=True)                                  # 60 deg rotation around z-axis
# r = rotate.from_rotvec([0, 0, np.deg2rad(150)])                               # 150 deg rotation around z-axis
# r = rotate.from_rotvec([0, 0, np.deg2rad(-219)])                              # -219 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(180), 0])                               # 180 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(220), 0])                               # 220 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(-60), 0])                               # -60 deg rotation around y-axis
# r = rotate.from_rotvec([np.deg2rad(150), 0, 0])                               # 150 deg rotation around x-axis
# r = rotate.from_rotvec([0, np.deg2rad(60), np.deg2rad(30)])                   # 60 deg rotation around y-axis, 30 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(180), np.deg2rad(180)])                 # 180 deg rotation around y-axis, 180 deg rotation around z-axis
r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(60), np.deg2rad(90)])       # crazy rotation :o

# x: right is positive, y: further away is positive, z: up is positive

# create original geons
g1 = RectangularPrism(2, np.array([-1, -1, 0]))             # 45 deg angle front left, no z info
g2 = RectangularPrism(3, np.array([0, 0, -1]))              # down
# g1 = RectangularPrism(2, np.array([0.02, 0.0, 1.0]))      # 45 deg angle front left, no z info
# g2 = RectangularPrism(3, np.array([-0.02, 0.0, 2.0]))     # down
g3 = RectangularPrism(2, np.array([1, 1, 0]))               # 45 deg angle back right, no z info
g4 = RectangularPrism(1, np.array([1, -1, 0]))              # 45 deg angle front right, no z info

# create original object and relations
original_object = Object(
# target_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# # create target geons (same as original, but with rotation applied)
# g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# # g1 = RectangularPrism(2, r.apply(np.array([0.02, 0.0, 1.0])))
# # g2 = RectangularPrism(3, r.apply(np.array([-0.02, 0.0, 2.0])))
# g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# g4 = RectangularPrism(1, r.apply(np.array([1, -1, 0])))

# MIRRORED target object
g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
g4 = RectangularPrism(1, r.apply(np.array([-1, 1, 0])))

target_object = Object(
# original_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

In [4]:
# add time needed for representation
total_run_time = total_run_time + (2 * object_encoding_time)

In [5]:
# Create graph
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]], subplot_titles=("Original Object", "Target Object"))

add_object(fig, original_object, "Original Object", colour='blue', row=1, col=1)
add_object(fig, target_object, "Target Object", colour='red', row=1, col=2)

fig.update_layout(title='3D Vector Visualization', showlegend=False)

# 2. Landmarking

In [6]:
# calculate axis of rotation, direction of rotation, and total angular disparity
center_point = np.array([0,0,0])
axis_of_rotation, direction, angle = find_axis_of_rotation(original_object, target_object, center_coords=center_point)
total_angular_disparity = np.rad2deg(angle)

# add time needed for landmarking
total_run_time = total_run_time + production_time                                                                 # check geon
total_run_time = total_run_time + production_time + (total_angular_disparity * propositional_difficulty_time)     # check spatial connection

[-0.8804842  -0.09751017 -0.46393894]


In [7]:
# Create graph
axis_fig = go.Figure(data=[])

print(original_object.get_landmark_endpoints())

add_object(axis_fig, original_object, "Original Object", colour='blue')
add_object(axis_fig, target_object, "Target Object", colour='red')

add_landmarks(axis_fig, original_object.get_landmark_endpoints(), "Original Landmarks", colour='purple')
add_landmarks(axis_fig, target_object.get_landmark_endpoints(), "Target Landmarks", colour='orange')
add_point(axis_fig, center_point, "Rotation Point", colour='green')

add_axis(axis_fig, axis_of_rotation, scale=2)

axis_fig.update_layout(title='3D Vector Visualization')

[[ 1.41421356  1.41421356  0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.         -1.        ]]


# 3. Rotation

In [8]:
# prepare graphs

graph_size = 4

# define graph w/ axis animation
axis_animation_fig = go.Figure(
    data =  create_data_from_object(original_object, "Original Object", colour='blue')
            + create_data_from_landmarks(original_object.get_landmark_endpoints(), "Original Landmark", colour='purple')
            + create_data_from_axis(axis_of_rotation, scale=2)
            + create_data_from_object(target_object, "Target Object", colour='red')
            + create_data_from_landmarks(target_object.get_landmark_endpoints(), "Target Landmark", colour='orange')
            + create_data_from_point(center_point, "Rotation Point", colour='green'),
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-graph_size, graph_size], autorange=False),
            yaxis=dict(range=[-graph_size, graph_size], autorange=False),
            zaxis=dict(range=[-graph_size, graph_size], autorange=False),
            aspectmode='cube'
        ),
        title="Animated Rotation",
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play", method="animate", args=[None, frame_args(production_time*3)])]
        )]
    )
)
axis_animation = []

# define graph w/ overlap animation
original_centerpoint_vec = find_center_point_LWLC(original_object)
target_centerpoint_vec = find_center_point_LWLC(target_object)
copy_og_obj = copy.deepcopy(original_object)
copy_tar_obj = copy.deepcopy(target_object)

copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
copy_tar_obj.update_start_coords(copy_tar_obj.start_coords - target_centerpoint_vec)

overlap_animation_fig = go.Figure(
    data =  create_data_from_object(copy_og_obj, "Original Object", colour='blue')
            + create_data_from_object(copy_tar_obj, "Target Object", colour='red'),
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-graph_size, graph_size], autorange=False),
            yaxis=dict(range=[-graph_size, graph_size], autorange=False),
            zaxis=dict(range=[-graph_size, graph_size], autorange=False),
            aspectmode='cube'
        ),
        title="Animated Rotation",
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play", method="animate", args=[None, frame_args(production_time*3)])]
        )]
    )
)
overlap_animation = []

# get landmark vectors, angular disparity
original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
curr_step_angular_disparity = total_angular_disparity

loop_count = 0
while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or greatest_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()) > landmark_distance_threshold:     # checking cosine similarity and distance between landmarks

    # find best axis/direction of rotation, angular disparity
    axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=np.deg2rad(curr_step_angular_disparity), total_angular_disparity=total_angular_disparity)
    curr_step_angular_disparity = np.rad2deg(curr_step_angular_disparity)

    # calculate step size
    # step_size = max_step_size if (curr_step_angular_disparity > total_angular_disparity * decrease_begins) else (curr_step_angular_disparity * step_size_decrease)
    step_size = curr_step_angular_disparity * step_size_decrease
    step_size = min(max(step_size, min_step_size), max_step_size)
    # step_size = np.rad2deg(prev_angle) * step_size_decrease
    # print(step_size)

    r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis

    # apply rotation to original object
    original_object.rotate(r)

    # update original_landmark_vector
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()

    # add animation frame to graph
    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)

    original_centerpoint_vec = find_center_point_LWLC(original_object)
    copy_og_obj = copy.deepcopy(original_object)
    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

    loop_count += 1

    if loop_count > 100:
        break

    print(cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector))
    print(greatest_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()))

axis_animation_fig.frames = axis_animation
axis_animation_fig.show()
overlap_animation_fig.frames = overlap_animation
overlap_animation_fig.show()

[-0.81244909 -0.32126837 -0.48653172]
0.30891607157483086
2.351312703023856
[0.82668351 0.32916666 0.45633724]
0.4363622181363541
2.123464681813467
[0.83339974 0.33537151 0.43928444]
0.6264742525458276
1.7286428143585304
[0.83677808 0.34165818 0.42786929]
0.8197958587466545
1.2006802780202412
[0.83988196 0.34798027 0.41654296]
0.9505930717511341
0.6286934276664015
[0.84576868 0.35300596 0.40007766]
0.9870287420782654
0.32213361105894606
[0.85309342 0.35353388 0.3837257 ]
0.996639898311395
0.16395369318451247


# 4. Decision

In [9]:
# check overall similarity

are_same, run_time = same_object(original_object, target_object, object_angle_threshold, total_angular_disparity, production_time, propositional_difficulty_time)
total_run_time += run_time

if are_same:
    print("they same woooo")
else:
    print("BAD")
    # repeat rotation
    # repeat landmarking if needed --> skip for now

    # mayve do: how many repetitions before confidence needed? low confidence needed person: 0, high confidence needed person: 2-3







BAD
